# pyBIS Workshop Level 0 — From GUI to Automation
**120 minutes | Beginner friendly | BAM Lab Notebook example**

**Three successes:** (1) retrieve my openBIS data; (2) modify metadata and verify it in the GUI; (3) automate a repetitive task.

This notebook is adapted from the supplied `lab-notebook-pybis.ipynb` and the [BAM Lab Notebook tutorial](https://bamresearch.github.io/openBIS-data-store-tutorials/lab-notebook/). It deliberately **does not** reproduce the full tutorial. It uses a project called `Material_Synthesis`, collections named **Synthesis** and **Parameter 1**, and a few `EXPERIMENTAL_STEP` objects.

**Important:** Work only in your assigned training Space. The notebook creates a dedicated `pyBIS Workshop Practice` object for editing; do not modify or delete original GUI tutorial objects. No terminal commands are needed if the instructor has prepared the environment.

**Schedule:** 0–10 welcome; 10–25 connection/Jupyter; 25–40 check or prepare data; 40–60 retrieve; 60–75 explore/filter; 75–85 break; 85–100 modify; 100–115 automate/export; 115–120 wrap-up. Times are flexible; the hands-on work is the priority.

## Part A — Connect and prepare (guided)
### 1. Connect to openBIS
1. Open the BAM Data Store playground in your browser and log in.
2. In **Tools → User Profile**, copy your openBIS session token (as described in the source notebook).
3. Run the next cell with **Shift + Enter**. Paste the token into the hidden prompt. **Never paste a token into a saved code cell or share it.**

The instructor should preinstall `pybis`, `pandas`, and the notebook kernel. If your installation uses a different login mechanism, use the instructor-provided connection cell.

In [ ]:
from getpass import getpass
from pybis import Openbis
import pandas as pd

o = Openbis("https://playground.datastore.bam.de/")
o.set_token(getpass("Paste your openBIS session token (hidden): "))
print("pyBIS connection configured. Next, select your Space.")

### 2. Select your own Space
Uncomment the first line to see accessible Spaces. Replace the placeholder with your assigned Space code.

In [ ]:
# display(o.get_spaces())
SPACE_CODE = "YOUR_SPACE_CODE"  # <-- change this
assert SPACE_CODE != "YOUR_SPACE_CODE", "Enter your assigned Space code first."
myspace = o.get_space(SPACE_CODE)
print("Using Space:", myspace.code)

### 3. Choose your path
**Path A:** You completed the GUI tutorial. The notebook looks for the existing `Material_Synthesis` project and its named collections.

**Path B:** You did not complete the GUI tutorial. The notebook creates **only missing** minimal structures: a project, two collections and a few demonstration experimental steps.

Run the same cells for either path. **No need to decide manually.** The checks are based on actual objects, not a broad `try/except` that could hide permission errors. Ask the instructor if you see duplicates or missing permissions.

**Note:** The original notebook uses `Material_Synthesis`, collection type `Collection`, and object type `EXPERIMENTAL_STEP`. These codes and property names must be supported by the workshop instance.

In [ ]:
PROJECT_CODE = "Material_Synthesis"
PROJECT_ID = f"/{SPACE_CODE}/{PROJECT_CODE.upper()}"

# Locate existing projects without assuming their code capitalization.
existing_projects = list(myspace.get_projects())
project = next((p for p in existing_projects if p.code.upper() == PROJECT_CODE.upper()), None)

if project is None:
    print("Path B: creating the missing Material Synthesis project")
    project = myspace.new_project(
        code=PROJECT_CODE,
        description="Synthetic material synthesis training project for pyBIS workshop"
    )
    project.save()
else:
    print("Path A: reusing existing project")

print("Project:", project.identifier)

### 4. Ensure the two collections exist
The original notebook creates five collections. For these exercises we need only **Synthesis** and **Parameter 1**. This cell finds collections by their displayed `$name`, then creates missing ones. It uses fixed workshop codes rather than the original `helper.py` function, so the notebook is self-contained.

In [ ]:
def collections_by_name(project):
    return {c.props.get("$name"): c for c in project.get_collections()}

collection_names = {
    "Synthesis": "PYBIS_SYNTHESIS",
    "Parameter 1": "PYBIS_PARAMETER_1",
}

available = collections_by_name(project)
for display_name, collection_code in collection_names.items():
    if display_name not in available:
        print("Creating missing collection:", display_name)
        # Also avoid a code collision with an existing collection.
        used_codes = {c.code.upper() for c in project.get_collections()}
        if collection_code.upper() in used_codes:
            raise RuntimeError(f"Code {collection_code} already exists with another name. Ask the instructor.")
        new_collection = o.new_collection(
            code=collection_code, type="Collection",
            project=project.identifier, props={"$name": display_name}
        )
        new_collection.save()
        available = collections_by_name(project)

synthesis = available["Synthesis"]
parameter_1 = available["Parameter 1"]
print("Ready:", synthesis.identifier, "and", parameter_1.identifier)

### 5. Ensure there is data for the exercises
We keep the example deliberately small. The GUI tutorial may already contain **Material Synthesis** and **Macrostructure Characterization**. If either is absent, create a simple synthetic training object. A separate practice object is created under **Synthesis** so metadata edits never overwrite a participant's original tutorial step.

The notebook uses `$name` and the `experimental_step.experimental_description` property from your source notebook. If the instance requires other fields, the instructor should adjust the templates before the session.

In [ ]:
def objects_by_name(collection):
    return {s.props.get("$name"): s for s in collection.get_samples()}

def ensure_step(collection, name, description):
    found = objects_by_name(collection).get(name)
    if found is not None:
        print("Reusing:", name)
        return found
    print("Creating:", name)
    obj = o.new_sample(
        type="EXPERIMENTAL_STEP", collection=collection,
        props={"$name": name,
               "experimental_step.experimental_description": description}
    )
    obj.save()
    return obj

material_step = ensure_step(
    synthesis, "Material Synthesis", "Synthetic synthesis example for training"
)
macro_step = ensure_step(
    parameter_1, "Macrostructure Characterization",
    "Synthetic macrostructure characterization example for training"
)
practice_step = ensure_step(
    synthesis, "pyBIS Workshop Practice",
    "Practice record: safe to edit during the pyBIS workshop"
)
print("Data ready. Open your project in the GUI and confirm the collections and objects.")

**GUI checkpoint:** Open the Material Synthesis project. Can you find **Synthesis**, **Parameter 1**, and the three named steps? If yes, you are ready for the hands-on exercises.

---
## Part B — Hands-on challenges
### Challenge 1 — Retrieve your data (20 minutes)
**First success:** *I can execute Python code and retrieve my openBIS data.*

**Task:** Retrieve the project and print its collections. Then find one experimental step by its identifier. You only need to change one line in the challenge cell.

In [ ]:
# Ready-to-run example
print("Project:", project.identifier)
for collection in project.get_collections():
    print("Collection:", collection.props.get("$name"), "→", collection.identifier)

print("Example experimental step:", material_step.identifier)
retrieved = o.get_sample(material_step.identifier)
print("Retrieved name:", retrieved.props.get("$name"))

In [ ]:
# YOUR TURN: replace the identifier with macro_step.identifier
chosen_id = material_step.identifier  # <-- change this
chosen_object = o.get_sample(chosen_id)
print(chosen_object.identifier)
print(chosen_object.props.get("$name"))

**Check:** Does the printed object match the experimental step you can see in the GUI?

**Hint:** `macro_step.identifier` is the identifier of the Macrostructure Characterization object.

### Challenge 2 — Explore and filter (15 minutes)
**Task:** Build a table from the objects in our two collections, inspect it and filter to one collection. We use a small local table so no global search across other researchers' Spaces is needed.

In [ ]:
rows = []
for collection_name, collection in [("Synthesis", synthesis), ("Parameter 1", parameter_1)]:
    for obj in collection.get_samples():
        rows.append({
            "collection": collection_name,
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "type": obj.type,
        })

objects_table = pd.DataFrame(rows, columns=["collection", "identifier", "name", "type"])
display(objects_table)
print("Total objects:", len(objects_table))

In [ ]:
# YOUR TURN: try "Synthesis" and "Parameter 1"
collection_to_find = "Synthesis"  # <-- change this
filtered = objects_table[objects_table["collection"] == collection_to_find]
display(filtered)
print("Matching objects:", len(filtered))

**Hint:** The text must match a collection name exactly. Try changing `collection_to_find` and running the cell again.

### Challenge 3 — Modify metadata (15 minutes)
**Second success:** *I can change an existing object's metadata and see the result in the GUI.*

**Task:** Edit **only** the `pyBIS Workshop Practice` step. Preview the change, then explicitly choose to save it. Verify in the GUI.

**Safety:** The field below comes from the uploaded source notebook. The object is dedicated to workshop edits. If the property is not editable on your instance, ask the instructor rather than changing a different field.

In [ ]:
practice_step = o.get_sample(practice_step.identifier)
PROPERTY = "experimental_step.experimental_description"
print("Current description:", practice_step.props.get(PROPERTY))

new_description = "I updated this training record using pyBIS!"  # <-- personalize this
print("Proposed description:", new_description)

# Set to True only after you are happy with the proposed text.
SAVE_CHANGE = False  # <-- change to True
if SAVE_CHANGE:
    practice_step.props[PROPERTY] = new_description
    practice_step.save()
    print("Saved to openBIS")
else:
    print("Preview only — nothing saved")

In [ ]:
# Verify from a fresh read of the server.
verified = o.get_sample(practice_step.identifier)
print("Description currently in openBIS:", verified.props.get(PROPERTY))

**GUI checkpoint:** Refresh the practice object in the Lab Notebook. Do you see your updated description?

### Challenge 4 — Automate and export (15 minutes)
**Third success:** *I can automate a task that would otherwise require repetitive GUI operations.*

**Task:** Retrieve all experimental steps in the two selected collections and export a CSV report. This uses a `for` loop; you do not need to write the loop from scratch. Only change the filename or selected collection(s).

In [ ]:
# Choose the collections to include in the report.
selected_collections = [synthesis, parameter_1]  # <-- optional: use [synthesis]

report_rows = []
for collection in selected_collections:
    for obj in collection.get_samples():
        report_rows.append({
            "collection": collection.props.get("$name"),
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "description": obj.props.get("experimental_step.experimental_description"),
        })

report = pd.DataFrame(
    report_rows, columns=["collection", "identifier", "name", "description"]
)
display(report)

filename = "my_openbis_experimental_steps.csv"  # <-- customize
report.to_csv(filename, index=False)
print(f"Exported {len(report)} records to {filename}")

**Check:** Open the generated CSV from the Jupyter file browser. Does it contain the names you saw in the GUI?

**Bonus (optional):** Change `selected_collections` to `[parameter_1]`, rerun, and compare the number of rows. Try exporting only the columns `identifier` and `name`.

---
## Part C — Wrap-up (5 minutes)
- **Retrieve:** Which pyBIS command gets an existing experimental step?
- **Modify:** What does `.save()` do? Why should you verify in the GUI?
- **Automate:** Which manual GUI task could you replace with a loop and a CSV export?

**Takeaway:** You do not have to memorize Python. You can start with working code, change a few values, and automate a useful openBIS task.

### Instructor notes / optional hints
- **Path A vs B:** Cells 3–5 reuse existing objects by project code and displayed `$name`; only missing items are created. A participant who completed the GUI tutorial may still get a dedicated practice object.
- **Data isolation:** Assign each participant their own Space or a separate project; do not have participants edit the same practice object.
- **Compatibility:** Test the code against the installed pyBIS version and training instance. In particular verify `get_collections()`, `get_samples()`, the collection type code `Collection`, and the description property.
- **Potential name collision:** If participants already created a project with a different code, update `PROJECT_CODE` before the workshop. If a collection has the same code but a different name, resolve manually rather than guessing.
- **Authentication:** Tokens must be entered via a hidden prompt and never stored in notebook outputs. Clear notebook outputs before distribution.
- **Excluded on purpose:** `helper.py`, ZIP downloads, dataset uploads, deletion, additional collections and advanced dataset search from the source notebook.
- **Fallback:** If connection or object creation fails, have a prepopulated individual project ready so participants can start at Challenge 1.
- **Solutions:** Challenge 1 → `chosen_id = macro_step.identifier`; Challenge 2 → `collection_to_find = "Parameter 1"`; Challenge 3 → set `SAVE_CHANGE = True` after preview; Challenge 4 → `selected_collections = [synthesis]`.